# Объём хранения и качество — без повторного обучения

Читаем готовый `metrics_all_runs.csv`. Данные CAMS, GPU и обучение не нужны.

Считаем **объём числовых массивов**, а не фактический размер полного сжатого архива:
`байт/кадр = payload_bytes + scaling_bytes_per_frame + shared_bytes / N`.

Два сценария: **decoder** — всё необходимое для восстановления; **full_codec** — энкодер и декодер. Для AE можно дополнительно измерить существующий полный `best.pt`. Для UMAP+KNN известна только нижняя оценка (обучающие поля и embedding); полный объект не сохранён. Метаданные, заголовки контейнеров и программный код исключены. Квантизация не применяется.

Качество берётся из прежнего эксперимента validation/test. Разные N моделируют амортизацию общих данных, а не новые тестовые выборки.

In [ ]:
# При необходимости: %pip install numpy pandas matplotlib
from pathlib import Path
import sys
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'scripts/storage_accounting.py').exists()), None)
assert ROOT is not None, 'Откройте ноутбук внутри репозитория DeepCompreesion'
sys.path.insert(0, str(ROOT))
from scripts.storage_accounting import storage_tables, plot_storage

CONFIG = {
    'metrics_csv': str(ROOT / 'metrics_all_runs.csv'),
    # Папка предыдущего эксперимента, содержащая ArticleSAMAutoencoder_d64_days.../best.pt.
    # None: используем только расчётные размеры, никаких моделей не создаём.
    'checkpoint_root': None,
    'field_shape': [3, 96, 84],  # без batch/channel; соответствует предоставленному CSV
    'raw_value_bytes': 4,
    'weight_value_bytes': 4,
    'basis_value_bytes': 4,
    'projection_value_bytes': 4,
    'umap_target_value_bytes': 4,
    'umap_embedding_value_bytes': 4,
    'archive_frame_counts': [1000, 10000, 100000],
    'plot_split': 'test',  # можно заменить на validation
    'output_dir': str(ROOT / 'outputs/storage_accounting'),
}
# Размеры элементов выше — предположения для shared-массивов исходного эксперимента.
# Не меняйте их ради "квантизации": качество после квантизации нужно оценивать отдельно.
# Payload и нормировка берутся непосредственно из CSV, включая индексы DCT/Wavelet.

## Компоненты хранения

Размер AE проверяется по числу параметров в CSV. PCA: базис + среднее; SVD: базис; RandomProjection: псевдообратная матрица для декодера и матрица проекции для полного комплекта. TT-SVD учитывается по фактическому payload из CSV, а не по номинальному latent_dim.

Если baseline-модели не сохранены, измерить их сериализованный размер без повторного fit нельзя. Поэтому расчётные размеры явно помечены; UMAP — lower_bound.

In [ ]:
components, scenarios = storage_tables(CONFIG)
display(components.loc[components.split == CONFIG['plot_split'], [
    'method', 'latent_dim', 'tt_ranks', 'train_frames',
    'payload_bytes', 'scaling_bytes_per_frame',
    'decoder_shared_array_bytes', 'full_shared_array_bytes',
    'shared_size_status', 'measured_full_checkpoint_bytes',
]])
print('Исходный кадр:', int(components.raw_array_bytes_per_frame.iloc[0]), 'байт')
print('Найдено checkpoint-файлов:',
      components.loc[components.measured_full_checkpoint_bytes.notna(),
                     'measured_checkpoint_path'].nunique())
print('Сохранено в:', CONFIG['output_dir'])

## Качество с учётом общих данных

`estimated_compression_ratio` — размер исходного массива / оценка байт на кадр. Для UMAP это верхняя граница коэффициента сжатия, поскольку часть расходов неизвестна. В столбце `measured_weights_plus_estimated_codes_bytes_per_frame` измерены только веса полного AE; коды по-прежнему рассчитаны.

In [ ]:
N = CONFIG['archive_frame_counts'][1] if len(CONFIG['archive_frame_counts']) > 1 else CONFIG['archive_frame_counts'][0]
view = scenarios[(scenarios.split == CONFIG['plot_split']) &
                 (scenarios.archive_frames == N) & (scenarios.scope == 'decoder')]
display(view[[
    'method', 'latent_dim', 'tt_ranks', 'mse', 'mae', 'ssim', 'relative_l2',
    'estimated_bytes_per_frame', 'estimated_bits_per_grid_value',
    'estimated_compression_ratio', 'ratio_is_upper_bound',
]].sort_values(['method', 'estimated_bytes_per_frame']))

In [ ]:
figure_paths = plot_storage(scenarios, CONFIG)
print('Графиков сохранено:', len(figure_paths))
display(sorted(p.name for p in Path(CONFIG['output_dir']).iterdir() if p.is_file()))

## Файлы результата

- `storage_components.csv`: компоненты объёма и исходные метрики.
- `storage_scenarios.csv`: все N и оба сценария.
- `error_vs_bytes_validation.csv`, `error_vs_bytes_test.csv`: раздельные таблицы.
- `error_vs_bytes_*.png/pdf`: MSE от байт на кадр.
- `storage_protocol.json`: настройки, допущения и контрольные суммы исходного CSV и скрипта.

Для статьи эти результаты следует называть **оценкой объёма хранения массивов**. Полный измеренный размер архива требует реальных сохранённых кодов, метаданных и моделей. Меньшая ошибка при одинаковом latent_dim не обязательно означает меньшую ошибку при одинаковом объёме хранения.